# RoBERTa Embeddings + Binary Classification (Hate vs Normal)

HateXplain has three classes: **hate**, **offensive**, and **normal**.
This notebook focuses on the cleaner binary problem — **hate vs normal** — by skipping
offensive samples during encoding. The offensive class sits ambiguously between hate and
normal and makes the task harder; isolating the two poles gives a clearer signal.

Pipeline:
1. Load HateXplain data (train / valid / test splits)
2. Encode posts with RoBERTa → 768-dim embeddings (offensive samples excluded during encoding)
3. Train ML classifiers on the hate vs normal embeddings


In [1]:
import json
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
# Hugging Face transformers for pre-trained RoBERTa tokenizer and model
from transformers import RobertaTokenizer, RobertaModel
import torch
from sklearn.metrics import classification_report, accuracy_score
# ML classifiers that will be trained on the RoBERTa embeddings
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Use GPU if available, otherwise CPU for RoBERTa inference
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# RoBERTa-base produces 768-dimensional embeddings per token
EMBEDDING_DIM = 768

print(f"Device: {DEVICE}")

Device: cpu


In [3]:
def load_hatexplain_data(file_path):
    """Read JSONL file — each line is one JSON object representing a data sample."""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def reconstruct_text(post_tokens):
    """HateXplain stores text as token lists; join them back into a string."""
    return ' '.join(post_tokens)

# Load the three HateXplain splits (train, validation, test)
train_data = load_hatexplain_data('hatexplain/hatexplain_train.jsonl')
valid_data = load_hatexplain_data('hatexplain/hatexplain_validation.jsonl')
test_data = load_hatexplain_data('hatexplain/hatexplain_test.jsonl')

print(f"Original - Train: {len(train_data)}, Valid: {len(valid_data)}, Test: {len(test_data)}")

Original - Train: 15383, Valid: 1922, Test: 1924


In [ ]:
# Label mappings
# Offensive (majority_label == 1) is excluded from this notebook —
# encode_dataset skips those samples so X/y arrays contain only hate and normal.
LABEL_MAP_BINARY = {0: 'hate', 1: 'normal'}   # binary label used in this notebook
LABEL_MAP_3CLASS = {0: 'hate', 1: 'offensive', 2: 'normal'}  # original 3-class for reference


Binary - Train: 9132, Valid: 1141, Test: 1142
Label distribution (train): hate=0: 4748, normal=1: 4384


In [5]:
# Load the pre-trained RoBERTa tokenizer (subword BPE) and base model
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
# Base model without a classification head — we extract raw embeddings
roberta_model = RobertaModel.from_pretrained('roberta-base').to(DEVICE)
# Eval mode disables dropout so embeddings are deterministic
roberta_model.eval()
print("RoBERTa model loaded")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1810.63it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa model loaded


In [ ]:
def get_roberta_embedding(text, tokenizer, model, device,
                          pooling='cls', rationale_scores=None):
    """Extract a fixed-size 768-dim embedding from RoBERTa.

    pooling strategies
    ------------------
    'cls'
        Use the <s> (CLS) token at position 0. After all transformer layers this
        token has attended to every other token, giving a compressed whole-sequence
        summary. Simple and fast.

    'mean'
        Average the hidden states of every real (non-padding) token equally.
        Integrates signal from all tokens rather than relying on a single position.

    'rationale'
        Weighted mean pooling using HateXplain annotator evidence scores as weights.
        Tokens the annotators flagged as evidence (score > 0) contribute more;
        tokens they ignored contribute less.

        Why this helps for hate speech:
          "i love everyone except those dirty [slur]"
          CLS/mean spread weight across all tokens equally.
          Rationale weighting concentrates weight on "[slur]" (score=1.0) —
          the actual hatefulness signal — making the embedding more discriminative.

        Falls back to mean pooling when all rationale scores are zero
        (e.g. normal-labelled posts where annotators flagged nothing).
    """
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True, max_length=512, padding=True,
    )
    inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs_on_device)

    hidden         = outputs.last_hidden_state   # (1, seq_len, 768)
    attention_mask = inputs['attention_mask']

    if pooling == 'cls':
        return hidden[0, 0, :].cpu().numpy().flatten()

    if pooling == 'mean':
        mask    = attention_mask[0].unsqueeze(-1).float().to(device)
        summed  = (hidden[0] * mask).sum(dim=0)
        return (summed / mask.sum()).cpu().numpy().flatten()

    if pooling == 'rationale':
        # Build a per-subtoken weight vector from word-level annotator scores.
        # Each whitespace word may map to multiple BPE subwords; all inherit
        # the parent word's rationale score.
        words    = text.split()
        word_ids = []
        for word_idx, word in enumerate(words):
            n_subtokens = len(tokenizer.encode(word, add_special_tokens=False))
            word_ids.extend([word_idx] * n_subtokens)

        max_content_tokens = inputs['input_ids'].shape[1] - 2  # excludes <s> and </s>
        word_ids = word_ids[:max_content_tokens]

        scores = (rationale_scores
                  if rationale_scores and any(s > 0 for s in rationale_scores)
                  else None)

        subtoken_weights = [
            scores[w_idx] if scores and w_idx < len(scores) else 1.0
            for w_idx in word_ids
        ]

        # Full weight vector: <s>=0, content subtokens, </s>=0, [PAD]=0
        seq_len      = inputs['input_ids'].shape[1]
        full_weights = [0.0] + subtoken_weights + [0.0] * (seq_len - len(subtoken_weights) - 1)
        full_weights = full_weights[:seq_len]

        att           = attention_mask[0].float().to(device)
        weight_tensor = torch.tensor(full_weights, dtype=torch.float32).to(device) * att
        weight_sum    = weight_tensor.sum()

        if weight_sum == 0:
            # All weights zero (normal sample with no rationale annotations) → mean pool
            mask = att.unsqueeze(-1)
            return ((hidden[0] * mask).sum(0) / mask.sum()).cpu().numpy().flatten()

        w = weight_tensor.unsqueeze(-1)
        return ((hidden[0] * w).sum(0) / weight_sum).cpu().numpy().flatten()

    raise ValueError(f"Unknown pooling='{pooling}'. Use 'cls', 'mean', or 'rationale'.")


In [ ]:
def encode_dataset(data, tokenizer, model, device,
                   pooling='rationale', sample_size=None):
    """Encode posts into RoBERTa embeddings, skipping offensive samples.

    Offensive samples (majority_label == 1) are excluded so the resulting arrays
    contain only hate (label=0) and normal (label=1) samples.

    pooling
    -------
    'cls'       — <s> token hidden state at position 0.
    'mean'      — Mean-pool all real (non-padding) token hidden states.
    'rationale' — Weighted mean using HateXplain annotator evidence scores.
                  Falls back to mean pooling when all scores are zero.

    Returned fields per sample
    --------------------------
    id                : original HateXplain sample ID
    source            : 'gab' | 'twitter'
    text              : reconstructed post string
    word_count        : number of whitespace tokens in the original post
    embedding         : 768-dim RoBERTa embedding
    label             : 0 = hate, 1 = normal  (offensive excluded)
    agreement         : fraction of annotators who voted for the majority label
    majority_target   : most-cited community targeted by the post
    rationale_scores  : per-word evidence strength averaged across annotators
    highlighted_tokens: words flagged by annotators as evidence (score > 0)
    """
    if sample_size:
        data = data[:sample_size]

    results  = []
    n_skipped = 0

    for item in tqdm(data, desc=f"Encoding ({pooling})"):
        tokens = item['post_tokens']
        text   = reconstruct_text(tokens)

        # Majority vote across annotators
        labels         = item['annotators']['label']
        majority_label = max(set(labels), key=labels.count)

        # Skip offensive samples — this is a hate vs normal binary experiment
        if majority_label == 1:
            n_skipped += 1
            continue

        agreement = labels.count(majority_label) / len(labels)

        # Most frequently cited target community across all annotators
        all_targets      = [t for tl in item['annotators']['target'] for t in tl]
        specific_targets = [t for t in all_targets if t != 'None']
        majority_target  = max(set(specific_targets), key=specific_targets.count) if specific_targets else 'None'

        source = item['id'].split('_')[-1]
        if source not in ('gab', 'twitter'):
            source = 'unknown'

        # Filter mismatched rationale arrays before averaging.
        # Some samples have annotators whose rationale list length differs from
        # the post token count; np.mean requires a rectangular array, so we drop them.
        rationales       = item['rationales']
        valid_rationales = [r for r in rationales if len(r) == len(tokens)]
        if valid_rationales:
            rationale_scores = np.mean(valid_rationales, axis=0).tolist()
        else:
            rationale_scores = [0.0] * len(tokens)

        highlighted_tokens = [t for t, s in zip(tokens, rationale_scores) if s > 0]

        # Remap original labels to binary: hate (0) → 0, normal (2) → 1
        binary_label = 0 if majority_label == 0 else 1

        embedding = get_roberta_embedding(
            text, tokenizer, model, device,
            pooling=pooling,
            rationale_scores=rationale_scores,
        )

        results.append({
            'id':                 item['id'],
            'source':             source,
            'text':               text,
            'word_count':         len(tokens),
            'embedding':          embedding,
            'label':              binary_label,
            'agreement':          agreement,
            'majority_target':    majority_target,
            'rationale_scores':   rationale_scores,
            'highlighted_tokens': highlighted_tokens,
        })

    print(f"  Encoded {len(results)} samples  ({n_skipped} offensive samples excluded)")
    return results


# ── Encode all splits ─────────────────────────────────────────────────────────
# Set SAMPLE_SIZE to a small integer (e.g. 500) for a quick run; None = full dataset.
# POOLING controls which embedding strategy is used. Options: 'cls', 'mean', 'rationale'
SAMPLE_SIZE = None
POOLING     = 'rationale'

print(f"Encoding with pooling='{POOLING}'...")
train_results = encode_dataset(train_data, roberta_tokenizer, roberta_model, DEVICE,
                               pooling=POOLING, sample_size=SAMPLE_SIZE)
valid_results = encode_dataset(valid_data, roberta_tokenizer, roberta_model, DEVICE,
                               pooling=POOLING, sample_size=SAMPLE_SIZE)
test_results  = encode_dataset(test_data,  roberta_tokenizer, roberta_model, DEVICE,
                               pooling=POOLING, sample_size=SAMPLE_SIZE)

print(f"\nReady: {len(train_results)} train / {len(valid_results)} valid / {len(test_results)} test")


Encoding training data...


Encoding: 100%|██████████| 9132/9132 [06:23<00:00, 23.83it/s]


Encoding validation data...


Encoding: 100%|██████████| 1141/1141 [00:48<00:00, 23.55it/s]


Encoding test data...


Encoding: 100%|██████████| 1142/1142 [00:48<00:00, 23.60it/s]


In [ ]:
def prepare_data(results):
    """Stack embeddings into (n_samples, 768), labels into (n_samples,), word counts into a list."""
    X           = np.array([r['embedding']  for r in results])
    y           = np.array([r['label']      for r in results])
    word_counts = [r['word_count'] for r in results]
    return X, y, word_counts

X_train, y_train, train_word_counts = prepare_data(train_results)
X_valid, y_valid, valid_word_counts = prepare_data(valid_results)
X_test,  y_test,  test_word_counts  = prepare_data(test_results)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Label distribution (train): hate={sum(y_train==0)}, normal={sum(y_train==1)}")


X_train shape: (9132, 768)
X_test shape: (1142, 768)
Label distribution (train): hate=0: 4748, normal=1: 4384


In [9]:
# Generic helper: train a classifier on embeddings, predict on test set, report metrics
def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    # Per-class precision, recall, F1 for binary (hate vs normal)
    print(classification_report(y_test, y_pred, target_names=['hate', 'normal']))
    
    return accuracy_score(y_test, y_pred)

In [ ]:
# 1. XGBoost — gradient-boosted decision trees
# Ref: https://xgboost.readthedocs.io/en/stable/parameter.html
xgb_model = xgb.XGBClassifier(
    # --- General Parameters ---
    # booster='gbtree',     # Which booster to use. Options:
    #                       #   'gbtree'   — tree-based models (default, best for most tasks)
    #                       #   'dart'     — tree-based with dropout (reduces overfitting)
    #                       #   'gblinear' — linear model (like regularized logistic regression)
    
    # --- Tree Booster Parameters ---
    n_estimators=100,       # Number of boosting rounds (trees). Try: 50, 200, 500, 1000
                            #   More trees = better fit but slower & risk overfitting
    max_depth=6,            # Max tree depth [default=6]. Try: 3, 4, 8, 10. 0 = no limit
                            #   Shallow trees generalize better; deep trees capture complex patterns
                            #   Deeper trees aggressively consume memory
    learning_rate=0.1,      # Step size shrinkage [default=0.3, alias: eta]. Range: [0, 1]
                            #   Try: 0.01, 0.05, 0.1, 0.3
                            #   Lower = needs more trees but often better generalization
    random_state=42,
    eval_metric='logloss',  # Evaluation metric. Options for binary classification:
                            #   'logloss' — negative log-likelihood (default for classification)
                            #   'error'   — misclassification rate (threshold 0.5)
                            #   'error@t' — misclassification with custom threshold t
                            #   'auc'     — ROC area under curve
                            #   'aucpr'   — precision-recall area under curve
    # --- Regularization & Sampling ---
    # min_child_weight=1,   # Min sum of instance weight (hessian) in a child [default=1]
    #                       #   Try: 1, 3, 5, 10. Range: [0, inf]
    #                       #   Higher = more conservative, prevents learning noise
    # gamma=0,              # Min loss reduction for a split [default=0, alias: min_split_loss]
    #                       #   Try: 0, 0.1, 0.5, 1.0. Range: [0, inf]
    #                       #   Acts as pruning — higher gamma = fewer splits
    # subsample=1.0,        # Fraction of training samples per tree [default=1]. Range: (0, 1]
    #                       #   Try: 0.6, 0.8. Subsampling reduces overfitting
    # sampling_method='uniform',  # How to sample training instances. Options:
    #                             #   'uniform'        — equal probability (default)
    #                             #   'gradient_based' — proportional to gradient magnitude
    #                             #                      allows subsample as low as 0.1
    # colsample_bytree=1.0, # Fraction of features per tree [default=1]. Range: (0, 1]
    #                       #   Try: 0.6, 0.8. Lower = more regularization
    # colsample_bylevel=1.0,# Fraction of features per tree depth level [default=1]
    # colsample_bynode=1.0, # Fraction of features per split [default=1]
    #                       #   These three stack multiplicatively:
    #                       #   bytree=0.5 * bylevel=0.5 * bynode=0.5 with 768 features
    #                       #   = 96 features at each split
    # reg_alpha=0,          # L1 regularization on weights [default=0, alias: alpha]
    #                       #   Try: 0, 0.01, 0.1, 1. Encourages sparsity
    # reg_lambda=1,         # L2 regularization on weights [default=1, alias: lambda]
    #                       #   Try: 0.1, 1, 5, 10. Penalizes large weights
    # max_delta_step=0,     # Max delta step per leaf output [default=0, 0 = no constraint]
    #                       #   Try: 1-10 for imbalanced binary classification
    #                       #   Helps control updates when classes are very imbalanced
    # scale_pos_weight=1,   # Balance positive/negative weights [default=1]
    #                       #   For imbalanced classes, try: sum(negative) / sum(positive)
    #                       #   e.g., if 70% normal / 30% hate, try ~2.3
    # --- Tree Construction ---
    # tree_method='auto',   # Tree construction algorithm [default='auto']. Options:
    #                       #   'auto'   — same as 'hist'
    #                       #   'exact'  — exact greedy, enumerates all splits (slow but optimal)
    #                       #   'approx' — approximate greedy using quantile sketch
    #                       #   'hist'   — faster histogram-based approximate (recommended)
    # grow_policy='depthwise',  # How new nodes are added [default='depthwise']
    #                           #   'depthwise' — split nodes closest to root (standard)
    #                           #   'lossguide' — split node with highest loss change (like LightGBM)
    #                           #   Only works with hist/approx tree_method
    # max_leaves=0,         # Max leaf nodes [default=0, 0 = no limit]
    #                       #   Use with grow_policy='lossguide' for leaf-wise growth
    # max_bin=256,          # Max bins for histogram [default=256]. Only for hist/approx
    #                       #   More bins = better splits but slower
    # --- DART booster params (if booster='dart') ---
    # rate_drop=0.0,        # Dropout rate [default=0.0]. Range: [0.0, 1.0]
    # sample_type='uniform',# How dropped trees are selected: 'uniform' or 'weighted'
    # normalize_type='tree',# Normalization: 'tree' or 'forest'
)
xgb_acc = train_and_evaluate(xgb_model, X_train, y_train, X_test, y_test, "XGBoost")


=== XGBoost ===
Accuracy: 0.7154
              precision    recall  f1-score   support

        hate       0.71      0.76      0.74       594
      normal       0.72      0.67      0.69       548

    accuracy                           0.72      1142
   macro avg       0.72      0.71      0.71      1142
weighted avg       0.72      0.72      0.71      1142



In [ ]:
# 2. MLP (Multi-Layer Perceptron)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),  # 3 hidden layers. Try: (128,), (512, 256), (256, 128)
                                        #   More/wider layers = more capacity but risk overfitting
                                        #   Fewer layers = simpler model, faster training
    activation='relu',      # Activation function applied at each hidden neuron.
                            # Options:
                            #   'relu'     — most common, fast, works well in practice.
                            #                Can cause "dying neurons" (output stuck at 0)
                            #   'tanh'     — outputs in [-1, 1], good for centered data.
                            #                Slower than relu, can suffer vanishing gradients
                            #   'logistic' — sigmoid, outputs in [0, 1].
                            #                Vanishing gradient problem with deep networks
                            #   'identity' — no activation (linear). Only useful if data is
                            #                linearly separable; defeats purpose of deep net
    max_iter=500,           # Max training epochs. Try: 200, 500, 1000
                            #   More iters = converges better, but early_stopping helps prevent waste
    random_state=42,
    early_stopping=True,    # Stop if validation score stops improving — prevents overfitting
    validation_fraction=0.1,  # Fraction of training data used for early stopping. Try: 0.1, 0.15
    # --- Other useful params to experiment with ---
    # solver='adam',         # Optimization algorithm. Options:
    #                        #   'adam'  — adaptive learning rate, good default for most tasks
    #                        #   'sgd'   — stochastic gradient descent, needs tuning (lr, momentum)
    #                        #             but can generalize better with proper schedule
    #                        #   'lbfgs' — quasi-Newton method, works well on small datasets
    #                        #             deterministic, no mini-batches
    # learning_rate_init=0.001,  # Initial learning rate (for sgd/adam). Try: 0.0001, 0.001, 0.01
    #                            #   Too high = unstable; too low = slow convergence
    # alpha=0.0001,         # L2 regularization strength. Try: 0.0001, 0.001, 0.01, 0.1
    #                       #   Higher = stronger regularization, prevents overfitting
    # batch_size='auto',    # Mini-batch size. Try: 32, 64, 128, 256
    #                       #   Smaller = noisier updates (can help generalization), slower
    #                       #   Larger = smoother updates, faster but may overfit
    # dropout: MLPClassifier doesn't support dropout directly — use PyTorch/Keras for that
)
mlp_acc = train_and_evaluate(mlp_model, X_train, y_train, X_test, y_test, "MLP Neural Network")


=== MLP Neural Network ===
Accuracy: 0.7215
              precision    recall  f1-score   support

        hate       0.72      0.75      0.74       594
      normal       0.72      0.69      0.70       548

    accuracy                           0.72      1142
   macro avg       0.72      0.72      0.72      1142
weighted avg       0.72      0.72      0.72      1142



In [ ]:
# 2b. PyTorch MLP — same simple architecture, but with more activation choices
# sklearn's MLPClassifier only supports: relu, tanh, logistic, identity
# PyTorch gives access to modern activations like GELU, SiLU, LeakyReLU, Mish, etc.
# This is still a simple shallow classifier on top of pre-computed embeddings

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class SimpleMLP(nn.Module):
    def __init__(self, input_dim=768, hidden_dims=(256, 128, 64), num_classes=2,
                 activation='relu'):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        # Map string name to PyTorch activation module
        activations = {
            'relu':      nn.ReLU(),       # Fast, works well. Can have "dying neurons"
            'tanh':      nn.Tanh(),       # Outputs [-1,1]. Good for centered data
            'sigmoid':   nn.Sigmoid(),    # Outputs [0,1]. Vanishing gradients in deep nets
            'leaky_relu':nn.LeakyReLU(),  # Like ReLU but allows small negatives — fixes dying neurons
            'gelu':      nn.GELU(),       # Used in BERT/RoBERTa/GPT. Smooth, state-of-art for NLP
            'silu':      nn.SiLU(),       # x*sigmoid(x), aka Swish. Smooth, often beats ReLU
            'mish':      nn.Mish(),       # x*tanh(softplus(x)). Strong in vision & NLP
            'elu':       nn.ELU(),        # Smooth for negatives. Faster convergence than ReLU
            'softplus':  nn.Softplus(),   # Smooth approximation of ReLU. No dead neurons
        }
        act_fn = activations[activation]
        
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(act_fn)
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

def train_pytorch_mlp(X_train, y_train, X_test, y_test, activation='gelu',
                       num_classes=2, target_names=['hate', 'normal'],
                       epochs=50, lr=1e-3, batch_size=64):
    """Train a PyTorch MLP and evaluate on test set."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    model = SimpleMLP(num_classes=num_classes, activation=activation).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(epochs):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_test, dtype=torch.float32).to(device)
        preds = model(X_t).argmax(dim=1).cpu().numpy()
    
    acc = accuracy_score(y_test, preds)
    print(f"\n=== PyTorch MLP (activation={activation}) ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=target_names))
    return acc

# Try GELU — the activation used inside RoBERTa/BERT itself
# Change 'gelu' to any of: relu, tanh, sigmoid, leaky_relu, silu, mish, elu, softplus
pytorch_mlp_acc = train_pytorch_mlp(X_train, y_train, X_test, y_test, activation='gelu')

In [ ]:
# 3. Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,          # Max solver iterations. Try: 500, 1000, 5000
                            #   Increase if you get convergence warnings
    random_state=42,
    # --- Key params and their alternatives ---
    # penalty='l2',         # Regularization type. Options:
    #                       #   'l2'         — ridge, shrinks all weights (default, usually best)
    #                       #   'l1'         — lasso, drives some weights to exactly 0 (feature selection)
    #                       #   'elasticnet' — mix of l1 + l2 (needs solver='saga', l1_ratio param)
    #                       #   None         — no regularization, risks overfitting on high-dim data
    # C=1.0,                # Inverse regularization strength. Try: 0.01, 0.1, 1, 10, 100
    #                       #   Smaller C = stronger regularization (simpler model)
    #                       #   Larger C  = weaker regularization (fits training data more closely)
    #                       #   With 768-dim embeddings, try C=0.1 or C=0.01 to reduce overfitting
    # solver='lbfgs',       # Optimization algorithm. Options:
    #                       #   'lbfgs'     — good default, supports l2/None (quasi-Newton)
    #                       #   'liblinear' — good for small datasets, supports l1/l2
    #                       #   'saga'      — scalable, supports all penalties incl. elasticnet
    #                       #   'newton-cg'  — good for multiclass, supports l2/None
    #                       #   'sag'       — fast on large datasets, supports l2/None
    # class_weight=None,    # Handle imbalanced classes. Options:
    #                       #   None        — treat all classes equally
    #                       #   'balanced'  — auto-weight inversely proportional to class frequency
    #                       #                 useful if hate samples are underrepresented
)
lr_acc = train_and_evaluate(lr_model, X_train, y_train, X_test, y_test, "Logistic Regression")


=== Logistic Regression ===
Accuracy: 0.7285
              precision    recall  f1-score   support

        hate       0.73      0.76      0.74       594
      normal       0.73      0.69      0.71       548

    accuracy                           0.73      1142
   macro avg       0.73      0.73      0.73      1142
weighted avg       0.73      0.73      0.73      1142



In [ ]:
# 4. Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,       # Number of trees in the forest. Try: 50, 200, 500, 1000
                            #   More trees = more stable predictions, diminishing returns past ~500
                            #   Unlike XGBoost, RF trees are independent so more trees won't overfit
    max_depth=10,           # Max depth per tree. Try: 5, 10, 20, None (unlimited)
                            #   None = trees grow until leaves are pure. Good with enough samples
                            #   Shallower = more regularization, less overfitting
    random_state=42,
    # --- Other useful params to experiment with ---
    # min_samples_split=2,  # Min samples required to split a node. Try: 2, 5, 10
    #                       #   Higher = more conservative splits, reduces overfitting
    # min_samples_leaf=1,   # Min samples in a leaf node. Try: 1, 2, 5, 10
    #                       #   Higher = smoother predictions, prevents learning noise
    # max_features='sqrt',  # Number of features considered per split. Options:
    #                       #   'sqrt'  — sqrt(768) ≈ 28 features per split (default, good for diversity)
    #                       #   'log2'  — log2(768) ≈ 10 features (more diversity between trees)
    #                       #   None    — all 768 features (less diversity, may overfit)
    #                       #   0.5     — 50% of features
    #                       #   sqrt is usually best — forces trees to be different from each other
    # criterion='gini',     # Split quality measure. Options:
    #                       #   'gini'    — Gini impurity (default, slightly faster)
    #                       #   'entropy' — information gain (slightly different splits)
    #                       #   'log_loss'— log loss, similar to entropy
    #                       #   In practice gini and entropy give very similar results
    # class_weight=None,    # Handle imbalanced classes. Options:
    #                       #   None               — treat all classes equally
    #                       #   'balanced'          — auto-weight by class frequency
    #                       #   'balanced_subsample' — rebalance per bootstrap sample
    # bootstrap=True,       # Whether to bootstrap (sample with replacement). Options:
    #                       #   True  — default, each tree sees a different ~63% of data
    #                       #   False — each tree sees all data (less diversity)
)
rf_acc = train_and_evaluate(rf_model, X_train, y_train, X_test, y_test, "Random Forest")


=== Random Forest ===
Accuracy: 0.6874
              precision    recall  f1-score   support

        hate       0.66      0.81      0.73       594
      normal       0.73      0.55      0.63       548

    accuracy                           0.69      1142
   macro avg       0.70      0.68      0.68      1142
weighted avg       0.69      0.69      0.68      1142



In [ ]:
print("\n" + "="*50)
print("SUMMARY - Binary Classification (Hate vs Normal)")
print("="*50)
print(f"XGBoost:          {xgb_acc:.4f}")
print(f"MLP (sklearn):    {mlp_acc:.4f}")
print(f"MLP (PyTorch):    {pytorch_mlp_acc:.4f}")
print(f"Logistic Reg:     {lr_acc:.4f}")
print(f"Random Forest:    {rf_acc:.4f}")


SUMMARY - Binary Classification (Hate vs Normal)
XGBoost:          0.7154
MLP:              0.7215
Logistic Reg:     0.7285
Random Forest:    0.6874


In [15]:
# Save embeddings
os.makedirs('embeddings', exist_ok=True)

np.save('embeddings/X_train_roberta_binary.npy', X_train)
np.save('embeddings/y_train_binary.npy', y_train)
np.save('embeddings/X_valid_roberta_binary.npy', X_valid)
np.save('embeddings/y_valid_binary.npy', y_valid)
np.save('embeddings/X_test_roberta_binary.npy', X_test)
np.save('embeddings/y_test_binary.npy', y_test)

print("Binary embeddings saved to embeddings/ folder")

Binary embeddings saved to embeddings/ folder
